<a href="https://colab.research.google.com/github/Aman6665/NLP-Program-using-Python-NLTK-and-spaCy/blob/main/24_Fine_Tuning_BERT_for_text_classification_using_Hugging_Face.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-Tuning BERT for Text Classification with Hugging Face

In this notebook, we will fine-tune a pre-trained BERT model (`bert-base-uncased`) on the IMDB dataset for sentiment classification (positive/negative reviews).

In [ ]:
!pip install -q transformers datasets accelerate evaluate scikit-learn

### 1. Load the Dataset
We will use a small subset of the IMDB dataset from Hugging Face to make sure training is fast and runs smoothly within standard Colab environments.

In [ ]:
from datasets import load_dataset

# Load the IMDB dataset
raw_datasets = load_dataset("imdb")

# Split into smaller subsets for demonstration purposes (to speed up training)
train_dataset = raw_datasets["train"].shuffle(seed=42).select(range(1000))
test_dataset = raw_datasets["test"].shuffle(seed=42).select(range(500))

print(train_dataset[0])

### 2. Tokenize the Text
We load the pre-trained BERT tokenizer to convert our text data into the token IDs and attention masks that BERT expects.

In [ ]:
from transformers import AutoTokenizer

model_checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

### 3. Load the Model and Define Metrics
We initialize the `AutoModelForSequenceClassification` with BERT and prepare our evaluation metric (Accuracy).

In [ ]:
from transformers import AutoModelForSequenceClassification
import evaluate
import numpy as np

# Load BERT model with 2 output labels (positive, negative)
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=2)

# Load accuracy metric
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

### 4. Setup Training Arguments and Trainer
Now we define our hyperparameters inside `TrainingArguments` and pass everything to the `Trainer` instance.

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

# Start fine-tuning
trainer.train()

### 5. Evaluate the Model
Once training is complete, we run final evaluation on the test set and display the metrics.

In [ ]:
eval_results = trainer.evaluate()
print("Evaluation Results:", eval_results)